# World Cup 2026 Bluesky Firehose — Consumer
**Team:** Bojan Ivanovski, Andrada Demian

This notebook is the **consumer** half of the project.

- **Ingestion (cells 0–4):** TCP receiver thread pulls newline-delimited JSON from
  the producer, micro-batches every 5 s into Spark, registers the **`raw`** temp
  view and appends to Parquet on disk.
- **Processing & visualisation (cells 6–13):** 60 s / 5 s windowed aggregations
  (post volume, mentioned users, referenced posts, external URLs), corpus-wide
  TF-IDF top-10, hashtag leaderboard, language distribution, time range, and
  optional VADER sentiment. All metrics are unioned into a single **`metrics`**
  temp view and written to CSV under `data/metrics/`. Charts are rendered inline;
  a live-updating Streamlit dashboard is in `viz_app.py`.

In [1]:

#  Imports
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import socket, json, threading, os
from datetime import datetime, timezone
from queue import Queue, Empty

RAW_OUTPUT_DIR = "./data/raw"
os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)

PRODUCER_HOST = "localhost"   # change if producer runs on another machine
PRODUCER_PORT = 9999

In [2]:
# Spark Session
spark = SparkSession.builder \
    .appName("WorldCupFirehose") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("[Spark] Session ready.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 12:39:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[Spark] Session ready.


In [3]:
# Schema definition (mirrors producer's output)
RAW_SCHEMA = StructType([
    StructField("did",         StringType(),  True),
    StructField("post_uri",    StringType(),  True),
    StructField("text",        StringType(),  False),
    StructField("created_at",  StringType(),  True),   # ISO 8601 string
    StructField("ingested_at", StringType(),  True),
    StructField("reply_to",    StringType(),  True),   # nullable
    StructField("mentions",    ArrayType(StringType()), True),
    StructField("urls",        ArrayType(StringType()), True),
    StructField("hashtags",    ArrayType(StringType()), True),
    StructField("lang",        StringType(),  True),
])

In [4]:
# TCP receiver thread
# Reads newline-delimited JSON from the producer and puts parsed dicts
# into a thread-safe queue for Spark to consume in batches.

post_queue = Queue()

def tcp_receiver():
    while True:
        try:
            print(f"[Consumer] Connecting to producer at {PRODUCER_HOST}:{PRODUCER_PORT}...")
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.connect((PRODUCER_HOST, PRODUCER_PORT))
                print("[Consumer] Connected to producer.")
                buffer = ""
                while True:
                    chunk = s.recv(4096).decode("utf-8")
                    if not chunk:
                        break
                    buffer += chunk
                    while "\n" in buffer:
                        line, buffer = buffer.split("\n", 1)
                        line = line.strip()
                        if line:
                            try:
                                post_queue.put(json.loads(line))
                            except json.JSONDecodeError:
                                pass
        except Exception as e:
            print(f"[Consumer] Connection error: {e}. Retrying in 5s...")
            import time; time.sleep(5)

receiver_thread = threading.Thread(target=tcp_receiver, daemon=True)
receiver_thread.start()
print("[Consumer] Receiver thread started.")

[Consumer] Connecting to producer at localhost:9999...
[Consumer] Receiver thread started.
[Consumer] Connected to producer.


In [5]:
# Micro-batch ingestion loop
# Every BATCH_INTERVAL seconds, drains the queue, creates a Spark DataFrame,
# registers/refreshes the "raw" temp view, and appends to disk as Parquet.

import time

BATCH_INTERVAL = 5      # seconds between micro-batches
batch_number   = 0
all_rows       = []     # accumulates all rows for the global "raw" table

print("[Consumer] Starting ingestion loop. Stop the cell to halt.")

while True:
    time.sleep(BATCH_INTERVAL)

    # Drain everything currently in the queue
    batch_rows = []
    while True:
        try:
            batch_rows.append(post_queue.get_nowait())
        except Empty:
            break

    if not batch_rows:
        print(f"[Batch {batch_number}] No new posts yet...")
        continue

    batch_number += 1
    all_rows.extend(batch_rows)

    # Build Spark DataFrame for this batch
    batch_df = spark.createDataFrame(batch_rows, schema=RAW_SCHEMA)

    # Register the FULL corpus as the "raw" temp view
    full_df = spark.createDataFrame(all_rows, schema=RAW_SCHEMA)
    full_df.createOrReplaceTempView("raw")

    # Append this batch to disk (Parquet, partitioned by date)
    batch_df.write \
        .mode("append") \
        .parquet(RAW_OUTPUT_DIR)

    print(f"[Batch {batch_number}] +{len(batch_rows)} posts | {len(all_rows)} total in raw | saved to {RAW_OUTPUT_DIR}")

[Consumer] Starting ingestion loop. Stop the cell to halt.


26/06/02 12:39:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/02 12:39:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/02 12:39:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[Batch 1] +8 posts | 8 total in raw | saved to ./data/raw


[Batch 2] +25 posts | 33 total in raw | saved to ./data/raw


[Batch 3] +17 posts | 50 total in raw | saved to ./data/raw


[Batch 4] +22 posts | 72 total in raw | saved to ./data/raw


[Batch 5] +10 posts | 82 total in raw | saved to ./data/raw


[Batch 6] +8 posts | 90 total in raw | saved to ./data/raw
[Batch 7] +14 posts | 104 total in raw | saved to ./data/raw


[Batch 8] +15 posts | 119 total in raw | saved to ./data/raw
[Batch 9] +16 posts | 135 total in raw | saved to ./data/raw


KeyboardInterrupt: 

---
## Processing & Visualisation
Runs **after** enough data has been collected by the ingestion loop above.
Cells can be re-run at any time to refresh metrics.

In [ ]:
# ── Processing: imports & output dirs ──────────────────────────────────────────
import os, time
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode, col, count, countDistinct, desc, lower,
    regexp_replace, split, to_timestamp, window, log, lit, avg,
)
from pyspark.sql.types import FloatType
from pyspark.ml.feature import StopWordsRemover

METRICS_DIR = "./data/metrics"
os.makedirs(METRICS_DIR, exist_ok=True)
print("[B] Imports OK.")

In [ ]:
# ── Load raw Parquet into a DataFrame and refresh the temp view ───────────────
def load_raw():
    return (
        spark.read.schema(RAW_SCHEMA).parquet(RAW_OUTPUT_DIR)
        .withColumn("ingested_ts", to_timestamp(col("ingested_at")))
        .withColumn("created_ts",  to_timestamp(col("created_at")))
    )

raw_df = load_raw()
raw_df.createOrReplaceTempView("raw")
print(f"[B] {raw_df.count()} posts loaded from {RAW_OUTPUT_DIR}")

In [ ]:
# ── Windowed aggregations: 60-second window / 5-second slide ──────────────────
WINDOW_DUR = "60 seconds"
SLIDE_DUR  = "5 seconds"

# Post volume per window
post_volume = (
    raw_df
    .groupBy(window("ingested_ts", WINDOW_DUR, SLIDE_DUR))
    .agg(count("*").alias("post_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "post_count",
    )
    .orderBy("window_start")
)

# Mention (user DID) occurrences per window
mention_counts = (
    raw_df
    .select("ingested_ts", explode("mentions").alias("mention"))
    .groupBy(window("ingested_ts", WINDOW_DUR, SLIDE_DUR), "mention")
    .agg(count("*").alias("occurrences"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "mention", "occurrences",
    )
    .orderBy(desc("occurrences"))
)

# Post-reference occurrences per window (which parent posts are being replied to)
post_reference_counts = (
    raw_df
    .filter(col("reply_to").isNotNull())
    .select("ingested_ts", col("reply_to").alias("ref_post"))
    .groupBy(window("ingested_ts", WINDOW_DUR, SLIDE_DUR), "ref_post")
    .agg(count("*").alias("occurrences"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "ref_post", "occurrences",
    )
    .orderBy(desc("occurrences"))
)

# URL occurrences per window
url_counts = (
    raw_df
    .select("ingested_ts", explode("urls").alias("url"))
    .groupBy(window("ingested_ts", WINDOW_DUR, SLIDE_DUR), "url")
    .agg(count("*").alias("occurrences"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "url", "occurrences",
    )
    .orderBy(desc("occurrences"))
)

post_volume.createOrReplaceTempView("post_volume")
mention_counts.createOrReplaceTempView("mention_counts")
post_reference_counts.createOrReplaceTempView("post_reference_counts")
url_counts.createOrReplaceTempView("url_counts")

print("[B] Windowed aggregations complete.")
post_volume.show(5, truncate=False)

In [ ]:
# ── TF-IDF: top-10 words across the full corpus ───────────────────────────────
EXTRA_STOP = {
    "rt", "amp", "https", "http", "t", "s", "co",
    "de", "la", "en", "el", "le", "les", "un", "une",
    "que", "et", "para", "con", "a", "e",
}
default_stops = set(StopWordsRemover.loadDefaultStopWords("english"))
all_stops = list(default_stops | EXTRA_STOP)

cleaned = raw_df.select(
    "post_uri",
    regexp_replace(lower(col("text")), r"[^a-z\s]", " ").alias("clean_text"),
)

# One row per (document, word)
words_per_doc = (
    cleaned
    .select("post_uri", explode(split(col("clean_text"), r"\s+")).alias("word"))
    .filter((col("word") != "") & (F.length("word") > 1))
    .filter(~col("word").isin(all_stops))
)

total_docs = float(raw_df.count())

# Corpus-level term frequency
tf = words_per_doc.groupBy("word").agg(count("*").alias("tf")).filter(col("tf") >= 2)

# Document frequency (number of distinct posts each word appears in)
df_counts = words_per_doc.groupBy("word").agg(countDistinct("post_uri").alias("df"))

tfidf_top10 = (
    tf.join(df_counts, "word")
    .withColumn("tfidf", col("tf") * log(lit(total_docs + 1) / (col("df") + 1)))
    .orderBy(desc("tfidf"))
    .limit(10)
)

tfidf_top10.createOrReplaceTempView("tfidf_top10")
print("[B] TF-IDF top-10:")
tfidf_top10.show(truncate=False)

In [ ]:
# ── Optional metrics ──────────────────────────────────────────────────────────

# Sentiment via VADER (graceful fallback if not installed)
HAS_SENTIMENT = False
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _vader = SentimentIntensityAnalyzer()

    @F.udf(returnType=FloatType())
    def _compound(text):
        return float(_vader.polarity_scores(text or "")["compound"])

    sentiment_df = raw_df.withColumn("sentiment", _compound(col("text")))
    avg_sent = sentiment_df.agg(avg("sentiment").alias("v")).first()["v"]
    print(f"[B] Average sentiment: {avg_sent:+.4f}  (-1 = very negative, +1 = very positive)")

    sentiment_over_time = (
        sentiment_df
        .groupBy(window("ingested_ts", WINDOW_DUR, SLIDE_DUR))
        .agg(avg("sentiment").alias("avg_sentiment"))
        .select(col("window.start").alias("window_start"), "avg_sentiment")
        .orderBy("window_start")
    )
    sentiment_over_time.createOrReplaceTempView("sentiment_over_time")
    HAS_SENTIMENT = True

except ImportError:
    print("[B] vaderSentiment not found — skipping sentiment.  pip install vaderSentiment")

# Hashtag leaderboard
hashtag_counts = (
    raw_df
    .select(explode(col("hashtags")).alias("hashtag"))
    .withColumn("hashtag", lower(col("hashtag")))
    .groupBy("hashtag")
    .agg(count("*").alias("occurrences"))
    .orderBy(desc("occurrences"))
    .limit(20)
)
hashtag_counts.createOrReplaceTempView("hashtag_counts")

# Language distribution
lang_dist = (
    raw_df
    .groupBy("lang")
    .agg(count("*").alias("post_count"))
    .orderBy(desc("post_count"))
)
lang_dist.createOrReplaceTempView("lang_dist")

# Time range covered by the dataset
tr = raw_df.select(
    F.min("created_ts").alias("earliest"),
    F.max("created_ts").alias("latest"),
).first()
print(f"[B] Time range: {tr['earliest']}  →  {tr['latest']}")
hashtag_counts.show(10, truncate=False)

In [ ]:
# ── Unified `metrics` temp view (long format) ─────────────────────────────────
# Union all windowed counter metrics into a single Spark temp view so downstream
# consumers can query one place. Schema:
#     metric_type | key      | window_start | window_end | value
# Per-corpus metrics (TF-IDF, hashtags, language distribution) stay in their
# own dedicated views since they have a different shape.

from pyspark.sql.functions import lit

metrics = (
    post_volume.select(
        lit("post_volume").alias("metric_type"),
        lit(None).cast("string").alias("key"),
        col("window_start"), col("window_end"),
        col("post_count").cast("double").alias("value"),
    )
    .unionByName(mention_counts.select(
        lit("mention").alias("metric_type"),
        col("mention").alias("key"),
        col("window_start"), col("window_end"),
        col("occurrences").cast("double").alias("value"),
    ))
    .unionByName(post_reference_counts.select(
        lit("post_reference").alias("metric_type"),
        col("ref_post").alias("key"),
        col("window_start"), col("window_end"),
        col("occurrences").cast("double").alias("value"),
    ))
    .unionByName(url_counts.select(
        lit("url").alias("metric_type"),
        col("url").alias("key"),
        col("window_start"), col("window_end"),
        col("occurrences").cast("double").alias("value"),
    ))
)

metrics.createOrReplaceTempView("metrics")
print("[B] Unified `metrics` view registered. Row counts by type:")
metrics.groupBy("metric_type").count().show()

In [ ]:
# ── Save all metrics to CSV (read by viz_app.py) ─────────────────────────────
def save_csv(df, name):
    path = f"{METRICS_DIR}/{name}.csv"
    df.toPandas().to_csv(path, index=False)
    print(f"  saved  {path}")

print("[B] Writing metrics to disk...")
save_csv(post_volume,           "post_volume")
save_csv(mention_counts,        "mention_counts")
save_csv(post_reference_counts, "post_reference_counts")
save_csv(url_counts,            "url_counts")
save_csv(tfidf_top10,           "tfidf_top10")
save_csv(hashtag_counts,        "hashtag_counts")
save_csv(lang_dist,             "lang_dist")
save_csv(metrics,               "metrics")
if HAS_SENTIMENT:
    save_csv(sentiment_over_time, "sentiment_over_time")

print("[B] Done. Temp views available: raw, metrics, post_volume, mention_counts,")
print("         post_reference_counts, url_counts, tfidf_top10, hashtag_counts, lang_dist")

In [ ]:
# ── Inline snapshot charts ────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle("World Cup 2026 Firehose — Metrics Snapshot", fontsize=14, fontweight="bold")

# Post volume over time
pv = post_volume.toPandas()
ax = axes[0, 0]
ax.bar(range(len(pv)), pv["post_count"], color="steelblue")
ax.set_xticks(range(len(pv)))
ax.set_xticklabels(
    pv["window_start"].astype(str).str[11:19],
    rotation=45, ha="right", fontsize=7
)
ax.set_title("Post volume (60 s windows)")
ax.set_ylabel("Posts")

# TF-IDF top-10 words
tw = tfidf_top10.toPandas().sort_values("tfidf")
ax = axes[0, 1]
bars = ax.barh(tw["word"], tw["tfidf"], color="tomato")
ax.set_title("TF-IDF top-10 words")
ax.set_xlabel("TF-IDF score")

# Top mentions (collapsed across all windows)
top_m = (
    mention_counts
    .groupBy("mention")
    .agg(F.sum("occurrences").alias("total"))
    .orderBy(desc("total"))
    .limit(10)
    .toPandas()
    .sort_values("total")
)
ax = axes[1, 0]
ax.barh(top_m["mention"], top_m["total"], color="seagreen")
ax.set_title("Top 10 mentioned users")
ax.set_xlabel("Occurrences")

# Top hashtags
hc = hashtag_counts.toPandas().head(10).sort_values("occurrences")
ax = axes[1, 1]
ax.barh(hc["hashtag"], hc["occurrences"], color="mediumpurple")
ax.set_title("Top 10 hashtags")
ax.set_xlabel("Occurrences")

plt.tight_layout()
snap = f"{METRICS_DIR}/snapshot.png"
plt.savefig(snap, dpi=120, bbox_inches="tight")
plt.show()
print(f"[B] Snapshot saved  {snap}")

In [ ]:
# ── Ad-hoc SQL examples ─────────────────────────────────────────────
spark.sql(
    "SELECT word, ROUND(tfidf, 3) AS score FROM tfidf_top10 ORDER BY score DESC"
).show()

spark.sql(
    "SELECT window_start, SUM(post_count) AS posts FROM post_volume"
    " GROUP BY window_start ORDER BY window_start DESC LIMIT 10"
).show(truncate=False)